# **Set Up**

In [ ]:
!pip install --pre deepchem
import deepchem
deepchem.__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 9.0 MB/s eta 0:00:00


wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


'2.8.1.dev'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from collections import OrderedDict

import deepchem as dc
import deepchem.models
import torch
from deepchem.models.torch_models import BasicMolGANModel as MolGAN
from deepchem.models.optimizers import ExponentialDecay
from torch.nn.functional import one_hot
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw

from deepchem.feat.molecule_featurizers.molgan_featurizer import GraphMatrix

# **Train**

In [ ]:
set(df.smiles.apply(lambda x: list(set([atom.GetAtomicNum() for atom in Chem.MolFromSmiles(x).GetAtoms()]))).sum())

{6, 7, 8, 9, 15, 16, 17, 35}

In [ ]:
# Specify the maximum number of atoms to encode for the featurizer and the MolGAN network.
# The higher the number of atoms, the more data you'll have in the dataset. However, this also increases the model complexity as the input dimensions become higher.
num_atoms = 12

# create featurizer - determine which atoms you want to include in the vocabulary (done by atomic number)
feat = dc.feat.MolGanFeaturizer(max_atom_count=num_atoms, atom_labels=[0, 6, 7, 8, 9, 15, 16, 17, 35]) #15, 16, 17, 19, 20, 24, 29, 35, 53, 80])
#feat = dc.feat.MolGanFeaturizer(max_atom_count=num_atoms, atom_labels=[0, 5, 6, 7, 8, 9, 11, 12, 13, 14]) #15, 16, 17, 19, 20, 24, 29, 35, 53, 80])
#feat = dc.feat.MolGanFeaturizer(max_atom_count=num_atoms, atom_labels=[0, 6, 7, 8, 9, 15, 16, 17, 35, 53]) #11, 12, 13, 14, 15, 16, 17, 19, 20, 24, 29, 35, 53, 80])

In [ ]:
# Process training data
training_data = 'docking_10000'
df = pd.read_csv('{}.csv'.format(training_data))
data = df[['smiles']]
smiles = data['smiles'].values

# Filter out SMILES longer than the desired length or invalid
filtered_smiles = []
for x in smiles:
  if Chem.MolFromSmiles(x) is None:
    break
  if Chem.MolFromSmiles(x).GetNumAtoms() < num_atoms:
    filtered_smiles.append(x)

len(filtered_smiles)

1082

In [ ]:
# featurize molecules
features = feat.featurize(filtered_smiles)
indices = [ i for i, data in enumerate(features) if type(data) is GraphMatrix ]
features = [features[i] for i in indices]
len(features)

1082

In [ ]:
# In case you want to save the training input (only the valid structures)
#from google.colab import files

#nmols = feat.defeaturize(features)
#nmols = list(filter(lambda x: x is not None, nmols))
#nmols_smiles = [Chem.MolToSmiles(m) for m in nmols]
#nmols_smiles_unique = list(OrderedDict.fromkeys(nmols_smiles))
#df = pd.DataFrame (nmols_smiles_unique, columns = ['smiles'])
#df.to_csv('MolGANInput.csv')
#files.download('MolGANInput.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# create model (set the learning rate and maximum number of atoms as the size of the vertices)
gan = MolGAN(learning_rate=ExponentialDecay(0.001, 0.9, 5000), vertices=num_atoms)
dataset = dc.data.NumpyDataset([x.adjacency_matrix for x in features],[x.node_features for x in features]) # Form for MolGAN to be happy

In [ ]:
def iterbatches(epochs):
    for i in range(epochs):
        for batch in dataset.iterbatches(batch_size=gan.batch_size, pad_batches=True):
            flattened_adjacency = torch.from_numpy(batch[0]).view(-1).to(dtype=torch.int64) # flatten the input because torch.nn.functional.one_hot only works with 1D inputs
            invalid_mask = (flattened_adjacency < 0) | (flattened_adjacency >= gan.edges) # edge type cannot be negative or >= gan.edges, these entries are invalid
            clamped_adjacency = torch.clamp(flattened_adjacency, 0, gan.edges-1) # clamp the input so it can be fed to the one_hot function
            adjacency_tensor = one_hot(clamped_adjacency, num_classes=gan.edges) # actual one_hot
            adjacency_tensor[invalid_mask] = torch.zeros(gan.edges, dtype=torch.long) # make the invalid entries, a vector of zeros
            adjacency_tensor = adjacency_tensor.view(*batch[0].shape, -1) # reshape to original shape and change dtype for stability.

            flattened_node = torch.from_numpy(batch[1]).view(-1).to(dtype=torch.int64)
            invalid_mask = (flattened_node < 0) | (flattened_node >= gan.nodes)
            clamped_node = torch.clamp(flattened_node, 0, gan.nodes-1)
            node_tensor = one_hot(clamped_node, num_classes=gan.nodes)
            node_tensor[invalid_mask] = torch.zeros(gan.nodes, dtype=torch.long)
            node_tensor = node_tensor.view(*batch[1].shape, -1)

            yield {gan.data_inputs[0]: adjacency_tensor, gan.data_inputs[1]:node_tensor}

In [ ]:
# Train the model
num_iterbatches = 25 # example had 25
num_generator_steps = 0.2 # example had 0.2
num_checkpoint_interval = 5000 # example had 5000
gan.fit_gan(iterbatches(25), generator_steps=0.2, checkpoint_interval=5000)

Ending global_step 549: generator average loss 31.1027, discriminator average loss -61.7381
TIMING: model fitting took 30.787 s


# **Predict**

In [ ]:
fragments = []

# Generate fragments
num_fragments = 50
generated_data = gan.predict_gan_generator(num_fragments)

# Convert to RDKit
nmols = feat.defeaturize(generated_data)

# Remove invalid molecules
nmols = list(filter(lambda x: x is not None, nmols))

# Keep unique SMILES
nmols_smiles = [Chem.MolToSmiles(m) for m in nmols]
nmols_smiles_unique = list(OrderedDict.fromkeys(nmols_smiles))

# Add this to the list of fragments
fragments += nmols_smiles_unique

# Show results
print('Success rate: {}/{}'.format(len(fragments), num_fragments))
fragments

Generating 50 samples
Success rate: 0/50


[21:34:46] Explicit valence for atom # 0 F, 3, is greater than permitted
[21:34:46] Explicit valence for atom # 3 F, 8, is greater than permitted
[21:34:46] Explicit valence for atom # 0 F, 3, is greater than permitted
[21:34:46] Explicit valence for atom # 0 O, 3, is greater than permitted
[21:34:46] Explicit valence for atom # 1 F, 5, is greater than permitted
[21:34:46] Explicit valence for atom # 1 O, 5, is greater than permitted
[21:34:46] Explicit valence for atom # 0 F, 3, is greater than permitted
[21:34:46] Explicit valence for atom # 0 F, 3, is greater than permitted
[21:34:46] Explicit valence for atom # 4 N, 7, is greater than permitted
[21:34:46] Explicit valence for atom # 0 N, 30, is greater than permitted
[21:34:46] Explicit valence for atom # 1 O, 5, is greater than permitted
[21:34:46] Explicit valence for atom # 0 F, 4, is greater than permitted
[21:34:46] Explicit valence for atom # 1 O, 5, is greater than permitted
[21:34:46] Explicit valence for atom # 0 F, 3, is 

[]

In [ ]:
#Export the created list as a csv file
from google.colab import files
generated_fragments = pd.DataFrame (fragments, columns = ['smiles'])
generated_fragments.to_csv('GAN_generated_fragments.csv')
files.download('GAN_generated_fragments.csv')